In [13]:
import tensorflow as tf
import keras
from keras import Input, Model 
from keras.layers import Dense, Flatten, Resizing

from keras.applications import vgg16, resnet50

from keras.datasets import cifar10



In [14]:
CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer',
         'dog', 'frog', 'horse', 'ship', 'truck']
INPUT_SHAPE = (32, 32, 3)

BACK_C = (.12, .12, .12, 1.)
plt.style.use('dark_background')

In [15]:
params = {'figure.facecolor': BACK_C,
          'axes.facecolor': BACK_C,
          'axes.titleweight': 'bold'}

plt.rcParams.update(params)

In [16]:
(X_train, y_train), (X_test, y_test) = cifar10.load_data()


# Transfert learning

>Le principe du *transfert learning* consiste à reprendre des réseaux déjà entraînés sur d'autres jeux de données afin de tranférer leur "apprentissage" à de nouveaux problèmes.
>
>En général, on entraîne des architectures qui ont déjà fait leur preuve sur de gros jeux de données très généralistes(*e.g.* [Coco](https://cocodataset.org/#home), [Imagenet](https://image-net.org/), *etc.*).
>
>L'objectif est d'utiliser les convolutions déjà entraînées afin d'extraire les features d'un jeu de donnée. Ceci a pour conséquence de radicalement réduire les temps d'entraînement puisqu'il n'y a plus le besoin d'entraîner la partie convolutive de nos CNN.

>Comment s'y prendre ?
>
>Il faut d'abord charger le modèle en prenant soin de fixer les paramètres comme suit:
>- include_top: permet de ne garder que la partie convolutive de VGG16. Dans notre cas, le *top* contient en fait les couches de sorties prévues pour traiter ImageNet ayant 1000 classes or nous n'avons que 10 classes en sortie.
>
>- input_shape: permet de definir la shape d'entrée, nous avons (32, 32, 3) mais par défaut VGG16 requiert des images de (224, 224, 3) et d'ailleurs a été entraîné avec cette shape. Nous y reviendrons.
>
>- weights: designe les poids pré-entraînés que l'on souhaite charger. Ici, ceux d'ImageNet.
>
>Il ne faut pas oublier de geler les couches de la partie permettant d'extraire les features (*i.e.* éviter de les réentraîner).
```py
INPUT_SHAPE = (32, 32, 3)

extractor = vgg16.VGG16(include_top=False, input_shape=INPUT_SHAPE, weights='imagenet')
extractor.trainable = False
```

>Ensuite, nous construisons notre modèle comme suit:
```py
inputs = Input(INPUT_SHAPE)

x = vgg16.preprocess_input(inputs)
x = extractor(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)

x = Dense(10, activation='softmax')(x)

model = Model(inputs, x)
```
>A noter qu'avant d'envoyer nos images dans notre extracteur, nous devons les faire passer par une fonction de preprocessing. Chaque architecture a sa propre fonction de preprocessing.



## VGG16 sans resizing

<br>

In [17]:
extractor = vgg16.VGG16(include_top=False, input_shape=INPUT_SHAPE, weights='imagenet')
extractor.trainable = False

inputs = Input(INPUT_SHAPE)

x = vgg16.preprocess_input(inputs)
x = extractor(x)

x = Flatten()(x)
x = Dense(256, activation='relu')(x)

x = Dense(10, activation='softmax')(x)

model = Model(inputs, x)

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=30, batch_size=1024, validation_split=.2)


Epoch 1/30
40/40 [==============================] - 4s 58ms/step - loss: 5.2684 - accuracy: 0.4586 - val_loss: 2.7625 - val_accuracy: 0.5421
Epoch 2/30
40/40 [==============================] - 1s 36ms/step - loss: 2.1055 - accuracy: 0.5728 - val_loss: 2.0676 - val_accuracy: 0.5595
Epoch 3/30
40/40 [==============================] - 1s 36ms/step - loss: 1.5145 - accuracy: 0.6191 - val_loss: 1.7850 - val_accuracy: 0.5784
Epoch 4/30
40/40 [==============================] - 1s 36ms/step - loss: 1.2096 - accuracy: 0.6583 - val_loss: 1.6637 - val_accuracy: 0.5890
Epoch 5/30
40/40 [==============================] - 1s 36ms/step - loss: 1.0127 - accuracy: 0.6939 - val_loss: 1.5813 - val_accuracy: 0.5997
Epoch 6/30
40/40 [==============================] - 1s 36ms/step - loss: 0.8842 - accuracy: 0.7229 - val_loss: 1.5385 - val_accuracy: 0.6071
Epoch 7/30
40/40 [==============================] - 1s 36ms/step - loss: 0.7763 - accuracy: 0.7486 - val_loss: 1.5022 - val_accuracy: 0.6067
Epoch 8/30
40

## Avec Resizing + data aug

>Les performances ne sont pas au rendez-vous. Rajoutons un peu de *data augmentation* et appliquons un *resize* afin d'avoir des images plus grandes et qui correspondent plus ce à quoi notre extractor est "habitué".
>
>Le fait d'agrandir les images devrait augmenter les temps d'entraînement mais nous permettra de converger plus rapidement (*i.e.* faire moins d'epochs).
>
>A noter également que nous pourrions nous contenter d'un *resize* plus sobre (*e.g.* (100, 100), (150, 150), *etc.*). 


In [18]:
### pre-trained model

extractor = vgg16.VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
extractor.trainable = False ## set to not trainable


### our model
### our data has shape INPUT_SHAPE=(32,32,3)
inputs = Input(INPUT_SHAPE)

x = Resizing(224, 224)(inputs) ## resize inputs to match the size of the pre-trained model
### specific preprocessing for VGG16
x = vgg16.preprocess_input(x) ## not trainable
### added augmentation to reduce overfitting
x = get_data_aug()(x)


### use pre-trained model
x = extractor(x) ## not trainable

### start custom/finetuning of model : trainable
x = Flatten()(x)
x = Dense(256, activation='relu')(x)

x = Dense(10, activation='softmax')(x)

### easy model definition because 1-tensorflow (tensorflow-tensorflow) model (not tensorflow-sklearn)
model = Model(inputs, x)

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=15, batch_size=256, validation_split=.2)


Epoch 1/15
157/157 [==============================] - 68s 388ms/step - loss: 2.1113 - accuracy: 0.6329 - val_loss: 0.6388 - val_accuracy: 0.7847
Epoch 2/15
157/157 [==============================] - 59s 378ms/step - loss: 0.6452 - accuracy: 0.7848 - val_loss: 0.5412 - val_accuracy: 0.8189
Epoch 3/15
157/157 [==============================] - 59s 379ms/step - loss: 0.5432 - accuracy: 0.8133 - val_loss: 0.4989 - val_accuracy: 0.8317
Epoch 4/15
157/157 [==============================] - 59s 378ms/step - loss: 0.4867 - accuracy: 0.8322 - val_loss: 0.4878 - val_accuracy: 0.8391
Epoch 5/15
157/157 [==============================] - 58s 373ms/step - loss: 0.4525 - accuracy: 0.8456 - val_loss: 0.4709 - val_accuracy: 0.8386
Epoch 6/15
157/157 [==============================] - 59s 373ms/step - loss: 0.4337 - accuracy: 0.8514 - val_loss: 0.4659 - val_accuracy: 0.8452
Epoch 7/15
157/157 [==============================] - 59s 374ms/step - loss: 0.4105 - accuracy: 0.8562 - val_loss: 0.4580 - val_ac

# ResNet

<br>

In [20]:
extractor = resnet50.ResNet50(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
extractor.trainable = False

inputs = Input(INPUT_SHAPE)

x = Resizing(224, 224)(inputs)
x = resnet50.preprocess_input(x)
x = get_data_aug()(x)

x = extractor(x)

x = Flatten()(x)
x = Dense(512, activation='relu')(x)
x = Dense(512, activation='relu')(x)

x = Dense(10, activation='softmax')(x)

model = Model(inputs, x)

model.compile(loss='sparse_categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

model.fit(X_train, y_train, epochs=15, batch_size=256, validation_split=.2)


Epoch 1/15
157/157 [==============================] - 42s 258ms/step - loss: 2.3092 - accuracy: 0.7277 - val_loss: 0.4746 - val_accuracy: 0.8462
Epoch 2/15
157/157 [==============================] - 40s 254ms/step - loss: 0.4973 - accuracy: 0.8316 - val_loss: 0.3951 - val_accuracy: 0.8687
Epoch 3/15
157/157 [==============================] - 40s 252ms/step - loss: 0.4451 - accuracy: 0.8499 - val_loss: 0.3555 - val_accuracy: 0.8863
Epoch 4/15
157/157 [==============================] - 39s 252ms/step - loss: 0.4018 - accuracy: 0.8642 - val_loss: 0.3788 - val_accuracy: 0.8781
Epoch 5/15
157/157 [==============================] - 39s 251ms/step - loss: 0.3624 - accuracy: 0.8779 - val_loss: 0.3942 - val_accuracy: 0.8783
Epoch 6/15
157/157 [==============================] - 39s 251ms/step - loss: 0.3535 - accuracy: 0.8792 - val_loss: 0.3549 - val_accuracy: 0.8947
Epoch 7/15
157/157 [==============================] - 40s 255ms/step - loss: 0.3268 - accuracy: 0.8890 - val_loss: 0.5007 - val_ac